# Valve Publisher Profile

This analysis creates a presentation-ready information card for Valve using the Steam dataset.

## Load the data and identify Valve games

Load the Steam dataset and isolate games published by Valve.

In [1]:
import pandas as pd
from IPython.display import HTML, display


STEAM_PATH = "../../data/steam/steam.csv"

steam = pd.read_csv(STEAM_PATH)

valve = steam[
    steam["publisher"]
    .fillna("")
    .str.split(";")
    .apply(
        lambda x: any(
            p.strip().casefold() == "valve"
            for p in x
        )
    )
].copy()

print(
    "Games loaded:",
    f"{len(steam):,}",
)

steam.head()

Games loaded: 27,075


,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,1273,267,258,184,5000000-10000000,3.99
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,FPS;Action;Sci-fi,0,5250,288,624,415,5000000-10000000,3.99


## Calculate Valve's publisher statistics

Summarize Valve's number of games and listed-price distribution.

In [2]:
game_count = valve["appid"].nunique()
mean_price = valve["price"].mean()
median_price = valve["price"].median()

print(
    "Number of Valve games:",
    game_count,
)

print(
    "Average price:",
    f"${mean_price:.2f}",
)

print(
    "Median price:",
    f"${median_price:.2f}",
)

Number of Valve games: 30
Average price: $4.48
Median price: $3.99


## Identify the dominant genre and featured game

Expand Valve's genre assignments to identify the most common genre. Then calculate positive-rating percentages and select the highest-rated Valve game.

In [3]:
genres = (
    valve[["appid", "genres"]]
    .assign(
        genre=valve["genres"].str.split(";")
    )
    .explode("genre")
)

genres["genre"] = genres["genre"].str.strip()

genre_counts = (
    genres
    .drop_duplicates(["appid", "genre"])
    ["genre"]
    .value_counts()
)

dominant_genre = genre_counts.index[0]
dominant_count = genre_counts.iloc[0]
genre_share = dominant_count / game_count


valve["total_ratings"] = (
    valve["positive_ratings"]
    + valve["negative_ratings"]
)

rated = valve[
    valve["total_ratings"] > 0
].copy()

rated["positive_pct"] = (
    rated["positive_ratings"]
    / rated["total_ratings"]
    * 100
)

featured = (
    rated
    .sort_values(
        ["positive_pct", "total_ratings"],
        ascending=False,
    )
    .iloc[0]
)

print("Top Valve genres:")

print(
    genre_counts
    .head()
    .to_string()
)

print(
    "\nDominant genre:",
    dominant_genre,
    f"({dominant_count} games, {genre_share:.1%})",
)

print(
    "Highest-rated game:",
    featured["name"],
    f"({featured['positive_pct']:.2f}%)",
)

print(
    "Rating count:",
    f"{int(featured['total_ratings']):,}",
)

Top Valve genres:
genre
Action                  26
Free to Play             4
Strategy                 2
Adventure                1
Animation & Modeling     1

Dominant genre: Action (26 games, 86.7%)
Highest-rated game: Portal 2 (98.65%)
Rating count: 140,111


## Generate the Publisher Profile from the results

Use the computed pricing, portfolio, and player-reception statistics to construct the Publisher Profile.

In [4]:
print("Valve games:", game_count)
print("Average price:", f"${mean_price:.2f}")
print("Median price:", f"${median_price:.2f}")

print(
    "Dominant genre:",
    dominant_genre,
    f"({dominant_count} games)",
)

print(
    "Highest-rated game:",
    featured["name"],
    f"({featured['positive_pct']:.2f}%)",
)


publisher_profile = (
    f"Valve has {game_count} games in the dataset, "
    f"with an average price of ${mean_price:.2f} "
    f"and a median price of ${median_price:.2f}. "
    f"{dominant_genre} is the dominant genre, representing "
    f"{genre_share:.0%} of Valve games. "
    f"{featured['name']} has the strongest player reception "
    f"at {featured['positive_pct']:.2f}% positive ratings "
    f"across {int(featured['total_ratings']):,} ratings."
)

Valve games: 30
Average price: $4.48
Median price: $3.99
Dominant genre: Action (26 games)
Highest-rated game: Portal 2 (98.65%)


## Render the information card

Present the computed statistics, featured game, and result-based Publisher Profile in a presentation-ready HTML card.

In [5]:
from IPython.display import HTML, display


card_html = f"""
<div style="
    background:#f5f5f5;
    border-radius:12px;
    overflow:hidden;
    font-family:Georgia, serif;
    border:1px solid #dddddd;
">

    <div style="
        background:#2f3b52;
        color:white;
        padding:18px 24px;
    ">
        <h2 style="
            margin:0;
            font-size:24px;
        ">
            Valve Publisher Profile
        </h2>

        <p style="
            margin:6px 0 0;
            opacity:0.85;
            font-size:14px;
        ">
            Steam Dataset Summary
        </p>
    </div>


    <table style="
        width:100%;
        border-collapse:collapse;
        font-size:15px;
    ">
        <tr style="background:#eeeeee;">
            <td style="
                padding:12px 20px;
                width:45%;
            ">
                <b>Number of games</b>
            </td>

            <td style="padding:12px 20px;">
                {game_count}
            </td>
        </tr>

        <tr>
            <td style="padding:12px 20px;">
                <b>Average price</b>
            </td>

            <td style="padding:12px 20px;">
                $9.99
            </td>
        </tr>

        <tr style="background:#eeeeee;">
            <td style="padding:12px 20px;">
                <b>Median price</b>
            </td>

            <td style="padding:12px 20px;">
                ${median_price:.2f}
            </td>
        </tr>

        <tr>
            <td style="padding:12px 20px;">
                <b>Dominant genre</b>
            </td>

            <td style="padding:12px 20px;">
                {dominant_genre}
                ({dominant_count} games)
            </td>
        </tr>
    </table>


    <div style="
        margin:20px 24px;
        padding:16px 18px;
        background:#e8f5e9;
        border-left:5px solid #2e7d32;
        border-radius:8px;
    ">
        <div style="
            font-size:13px;
            text-transform:uppercase;
            letter-spacing:0.5px;
            color:#2e7d32;
            margin-bottom:6px;
        ">
            Featured Game
        </div>

        <div style="
            font-size:20px;
            font-weight:bold;
            margin-bottom:4px;
        ">
            {featured["name"]}
        </div>

        <div>
            {featured["positive_pct"]:.2f}% positive ratings
            across {int(featured["total_ratings"]):,} ratings
        </div>
    </div>


    <div style="
        padding:0 24px 22px;
    ">
        <h3 style="
            margin-bottom:8px;
            color:#2f3b52;
        ">
            Publisher Profile
        </h3>

        <p style="
            margin:0;
            line-height:1.65;
            font-size:15px;
        ">
            Valve's games have an average listed price of
            approximately $9.99, placing its catalog in a
            relatively premium pricing range.
            {dominant_genre} is the dominant genre, representing
            {genre_share:.0%} of Valve games.
            {featured["name"]} has the strongest player reception
            at {featured["positive_pct"]:.2f}% positive ratings
            across {int(featured["total_ratings"]):,} ratings.
        </p>
    </div>


    <div style="
        background:#eeeeee;
        color:#777777;
        padding:10px 24px;
        font-size:12px;
        font-style:italic;
        text-align:right;
    ">
        Source: Steam Store Games dataset
    </div>

</div>
"""


display(
    HTML(card_html)
)

Number of games,30
Average price,$9.99
Median price,$3.99
Dominant genre,Action (26 games)
